# Выдача рейтенга фильма

In [1]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


CUDA: True
NVIDIA GeForce RTX 5070 Ti


### Конфиг (параметры обучения)

In [2]:
from pathlib import Path
import pandas as pd
import re
import os
import warnings
from typing import Dict, List, Optional

# Конфигурация путей
ROOT = Path(".").resolve() 
OUTPUT_DIR = ROOT / "Out"
DATA_DIR = ROOT / "datasets"
METADATA_CSV = DATA_DIR / "scripts_ratingss.csv"
PROCESSED_DIR = DATA_DIR / "Pipe"
RAW_DATA_DIR = DATA_DIR / "scenaryy"
TEST_TXT_DIR = DATA_DIR / "noinfo"
MORE_TEST_TXT_DIR = DATA_DIR / "wrong"

# Параметры
MIN_TEXT_LENGTH = 1000
MAX_TEXT_LENGTH = 260000
ENCODING = 'utf-8'

# Создание директорий
for dir_path in [RAW_DATA_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print("Конфигурация загружена!")
print(f"ROOT: {ROOT}")

Конфигурация загружена!
ROOT: C:\Users\Дмитрий\Downloads\Ratings


## Загрузка и проверка данных

In [3]:
class DataLoader:
    @staticmethod
    def load_metadata(csv_path: Path) -> pd.DataFrame:
        try:
            df = pd.read_csv(csv_path, encoding=ENCODING)
            print(f"Загружено {len(df)} записей из {csv_path.name}")
            return df
        except Exception as e:
            raise Exception(f"Ошибка загрузки CSV: {e}")
    
    @staticmethod
    def load_script_txt(file_path: Path) -> str:
        try:
            with open(file_path, 'r', encoding=ENCODING) as f:
                text = f.read()
            return text
        except UnicodeDecodeError:
            for encoding in ['cp1251', 'iso-8859-1', 'mac_cyrillic']:
                try:
                    with open(file_path, 'r', encoding=encoding) as f:
                        return f.read()
                except:
                    continue
            raise Exception(f"Не удалось декодировать файл: {file_path.name}")

class TextPreprocessor:
    @staticmethod
    def clean_script_text(raw_text: str) -> str:
        text = raw_text
        text = re.sub(r'\r\n', '\n', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[ \t]{2,}', ' ', text)
        text = re.sub(r'\([^)]*\)', '', text)
        text = re.sub(r'^[A-ZА-Я\s\d\-\.]+$', '', text, flags=re.MULTILINE)
        text = re.sub(r'[^\w\s\.,!?;:\-\'\"\(\)\n]', '', text)
        text = re.sub(r'\s+([.,!?;:])', r'\1', text)
        text = re.sub(r'([.,!?;:])\s+', r'\1 ', text)
        return text.strip()

# Загрузка метаданных
metadata_df = DataLoader.load_metadata(METADATA_CSV)
print("\nПервые 3 записи:")
print(metadata_df.head(3))

Загружено 53 записей из scripts_ratingss.csv

Первые 3 записи:
                                          filename  \
0                           8_миллиметров_Кино.txt   
1                        13_причин_почему_Кино.txt   
2  Kingsman_Секретная_служба_на_русском_читать.txt   

                                         title  year  kp_rating  imdb_rating  \
0                           8_миллиметров_Кино  1999        7.1          6.6   
1                        13_причин_почему_Кино  2017        7.3          7.4   
2  Kingsman_Секретная_служба_на_русском_читать  2015        7.7          7.7   

  notes age_rating_imdb age_rating_kp                  english_title  
0    ok               R           18+                   8 Millimeter  
1    ok           TV-MA     Not found           TH1RTEEN R3ASONS WHY  
2    ok       NOT_RATED           18+  Kingsman - The Secret Service  


## Подготовка к обучению

In [4]:
def prepare_training_data(metadata_df, raw_data_dir, processed_dir):
    processed_data = []
    
    for idx, row in metadata_df.iterrows():
        try:
            filename = str(row['filename'])
            if not filename.endswith('.txt'):
                filename += '.txt'
            
            txt_path = raw_data_dir / filename
            
            if not txt_path.exists():
                print(f"Файл не найден: {filename}")
                continue
            
            # Загрузка и очистка текста
            raw_text = DataLoader.load_script_txt(txt_path)
            cleaned_text = TextPreprocessor.clean_script_text(raw_text)
            
            if len(cleaned_text) < MIN_TEXT_LENGTH:
                continue
            
            # Создание записи для обучения
            record = {
                'filename': filename,
                'title': row['title'],
                'year': row['year'],
                'kp_rating': row['kp_rating'],
                'imdb_rating': row['imdb_rating'],
                'age_rating': row['age_rating_kp'] if pd.notna(row['age_rating_kp']) else row['age_rating_imdb'],
                'text': cleaned_text[:5000]  # Берем первые 5000 символов для обучения
            }
            processed_data.append(record)
            
        except Exception as e:
            print(f"Ошибка обработки файла {filename}: {e}")
            continue
    
    return pd.DataFrame(processed_data)

# Создание датасета
processed_df = prepare_training_data(metadata_df, RAW_DATA_DIR, PROCESSED_DIR)
print(f"\nСоздан датасет: {len(processed_df)} записей")


Создан датасет: 53 записей


In [4]:
from sklearn.model_selection import train_test_split

# Создание бинарных меток для рейтинга (выше/ниже медианы)
median_rating = processed_df['kp_rating'].median()
processed_df['rating_label'] = (processed_df['kp_rating'] > median_rating).astype(int)

# Преобразование возрастного рейтинга в числовые метки
age_mapping = {
    '18+': 0,
    '16+': 1,
    '12+': 2,
    '6+': 3,
    '0+': 4,
    'Not found': 5
}
processed_df['age_label'] = processed_df['age_rating'].map(age_mapping).fillna(5).astype(int)

# Разделение данных
train_df, val_df = train_test_split(
    processed_df,
    test_size=0.2,
    random_state=42,
    stratify=processed_df['rating_label']
)

print(f"\nРазделение данных:")
print(f"Тренировочная выборка: {len(train_df)} записей")
print(f"Валидационная выборка: {len(val_df)} записей")

NameError: name 'processed_df' is not defined

In [ ]:
def create_prompts(df, task_type="rating"):
    """Создание промптов для разных задач"""
    prompts = []
    
    for idx, row in df.iterrows():
        if task_type == "rating":
            # Промпт для предсказания рейтинга
            prompt = f"""Текст сценария фильма "{row['title']}" ({row['year']}):

{row['text']}

На основе этого текста сценария, оцени рейтинг фильма от 1 до 10.
Рейтинг:"""
            answer = f" {row['kp_rating']:.1f}"
            
        elif task_type == "age_rating":
            # Промпт для предсказания возрастного рейтинга
            prompt = f"""Текст сценария фильма "{row['title']}" ({row['year']}):

{row['text']}

Определи возрастной рейтинг для этого фильма (18+, 16+, 12+, 6+, 0+).
Возрастной рейтинг:"""
            answer = f" {row['age_rating']}"
        
        elif task_type == "both":
            # Промпт для предсказания обоих показателей
            prompt = f"""Текст сценария фильма "{row['title']}" ({row['year']}):

{row['text']}

Проанализируй сценарий и определи:
1. Рейтинг фильма от 1 до 10
2. Возрастной рейтинг (18+, 16+, 12+, 6+, 0+)

Ответ:"""
            answer = f" Рейтинг: {row['kp_rating']:.1f}, Возрастной рейтинг: {row['age_rating']}"
        
        prompts.append(prompt + answer)
    
    return prompts

# Создание промптов для разных задач
train_prompts_rating = create_prompts(train_df, "rating")
val_prompts_rating = create_prompts(val_df, "rating")

train_prompts_age = create_prompts(train_df, "age_rating")
val_prompts_age = create_prompts(val_df, "age_rating")

train_prompts_both = create_prompts(train_df, "both")
val_prompts_both = create_prompts(val_df, "both")

# Сохранение промптов
prompts_dir = PROCESSED_DIR / "prompts"
prompts_dir.mkdir(exist_ok=True)

with open(prompts_dir / "train_rating.txt", "w", encoding=ENCODING) as f:
    f.write("\n\n".join(train_prompts_rating))

with open(prompts_dir / "val_rating.txt", "w", encoding=ENCODING) as f:
    f.write("\n\n".join(val_prompts_rating))

print(f"\nПромпты сохранены в: {prompts_dir}")

# DL

In [ ]:

print("\n" + "="*50)
print("УСТАНОВКА И ЗАГРУЗКА МОДЕЛИ")
print("="*50)

# Установка библиотек (если нужно)
import sys
import subprocess
import importlib

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

required_packages = [
    "transformers>=4.41",
    "datasets",
    "peft",
    "accelerate",
    "bitsandbytes",
    "trl",
    "sentencepiece",
    "scikit-learn"
]

for package in required_packages:
    try:
        importlib.import_module(package.split('>=')[0].split('==')[0])
    except ImportError:
        print(f"Установка {package}...")
        install_package(package)

# Импорт библиотек
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import torch

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B"

# Конфигурация квантизации для экономии памяти
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Загрузка токенизатора
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    trust_remote_code=True
)

# Добавление pad token если его нет
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

### Train/Val split

In [ ]:
def prepare_dataset_for_finetuning(prompts_list):
    """Подготовка датасета для тонкой настройки"""
    dataset = Dataset.from_dict({"text": prompts_list})
    
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )
    
    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=["text"]
    )
    
    return tokenized_dataset

# Подготовка датасетов
train_dataset = prepare_dataset_for_finetuning(train_prompts_both)
val_dataset = prepare_dataset_for_finetuning(val_prompts_both)

print(f"\nРазмеры датасетов:")
print(f"Обучающий: {len(train_dataset)} примеров")
print(f"Валидационный: {len(val_dataset)} примеров")

### LoRA + настройка модели

In [ ]:
# Загрузка модели
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Подготовка модели для k-bit обучения
model = prepare_model_for_kbit_training(model)

# Конфигурация LoRA
lora_config = LoraConfig(
    r=8,  # Rank
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

# Применение LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Параметры обучения
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR / "model_output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=True,
    optim="paged_adamw_8bit",
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none"
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("\n" + "="*50)
print("НАЧАЛО ОБУЧЕНИЯ")
print("="*50)

# Обучение модели
trainer.train()

# Сохранение модели
model.save_pretrained(OUTPUT_DIR / "trained_model")
tokenizer.save_pretrained(OUTPUT_DIR / "trained_model")
print(f"\nМодель сохранена в: {OUTPUT_DIR / 'trained_model'}")


### Обучалка

In [ ]:
def predict_movie_ratings(text_file_path, model, tokenizer):
    """
    Предсказание рейтинга и возрастного рейтинга для нового сценария
    
    Args:
        text_file_path: Путь к текстовому файлу со сценарием
        model: Обученная модель
        tokenizer: Токенизатор
    """
    # Загрузка текста
    with open(text_file_path, 'r', encoding='utf-8') as f:
        script_text = f.read()
    
    # Очистка текста
    cleaned_text = TextPreprocessor.clean_script_text(script_text)
    
    # Создание промпта
    prompt = f"""Текст сценария фильма:

{cleaned_text[:3000]}

Проанализируй сценарий и определи:
1. Рейтинг фильма от 1 до 10
2. Возрастной рейтинг (18+, 16+, 12+, 6+, 0+)

Ответ:"""
    
    # Токенизация
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    
    # Генерация ответа
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids.cuda(),
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            top_p=0.9
        )
    
    # Декодирование ответа
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Извлечение ответа (только часть после промпта)
    generated_text = answer[len(prompt):].strip()
    
    # Парсинг ответа
    rating = None
    age_rating = None
    
    # Поиск рейтинга
    rating_match = re.search(r'Рейтинг:\s*([0-9.]+)', generated_text)
    if rating_match:
        rating = float(rating_match.group(1))
    
    # Поиск возрастного рейтинга
    age_match = re.search(r'Возрастной рейтинг:\s*([0-9+]+)', generated_text)
    if not age_match:
        age_match = re.search(r'([0-9]+\+)', generated_text)
    
    if age_match:
        age_rating = age_match.group(1)
    
    return {
        "filename": text_file_path.name,
        "predicted_rating": rating,
        "predicted_age_rating": age_rating,
        "full_response": generated_text
    }

In [ ]:
print("\n" + "="*50)
print("ТЕСТИРОВАНИЕ НА НОВЫХ ФАЙЛАХ")
print("="*50)

# Тестирование на файлах из тестовых директорий
test_dirs = [TEST_TXT_DIR, MORE_TEST_TXT_DIR]

for test_dir in test_dirs:
    if test_dir.exists():
        print(f"\nТестирование файлов из: {test_dir}")
        
        # Поиск всех txt файлов
        test_files = list(test_dir.glob("*.txt"))
        
        for test_file in test_files[:3]:  # Тестируем первые 3 файла
            try:
                result = predict_movie_ratings(test_file, model, tokenizer)
                print(f"\nФайл: {result['filename']}")
                print(f"Предсказанный рейтинг: {result['predicted_rating']}")
                print(f"Предсказанный возрастной рейтинг: {result['predicted_age_rating']}")
                print(f"Полный ответ: {result['full_response'][:100]}...")
            except Exception as e:
                print(f"Ошибка при обработке {test_file.name}: {e}")

In [ ]:
def interactive_testing():
    """Интерактивный режим тестирования"""
    print("\n" + "="*50)
    print("ИНТЕРФЕКТИВНЫЙ РЕЖИМ ТЕСТИРОВАНИЯ")
    print("="*50)
    print("Введите путь к текстовому файлу со сценарием (или 'exit' для выхода)")
    
    while True:
        file_path = input("\nПуть к файлу: ").strip()
        
        if file_path.lower() == 'exit':
            break
        
        file_path = Path(file_path)
        
        if not file_path.exists():
            print(f"Файл не найден: {file_path}")
            continue
        
        try:
            result = predict_movie_ratings(file_path, model, tokenizer)
            print(f"\n{'='*40}")
            print(f"РЕЗУЛЬТАТЫ ДЛЯ: {result['filename']}")
            print(f"{'='*40}")
            print(f"📊 Предсказанный рейтинг: {result['predicted_rating']:.1f}/10")
            print(f"🎭 Возрастной рейтинг: {result['predicted_age_rating']}")
            print(f"\n📝 Анализ модели:")
            print(f"{result['full_response']}")
            print(f"{'='*40}")
            
        except Exception as e:
            print(f"Ошибка: {e}")

# Запуск интерактивного режима (раскомментировать при необходимости)
# interactive_testing()

# ==================================================
# ОЦЕНКА КАЧЕСТВА МОДЕЛИ
# ==================================================
def evaluate_model(model, tokenizer, test_df):
    """Оценка точности модели на тестовых данных"""
    correct_rating = 0
    correct_age = 0
    total = len(test_df)
    
    predictions = []
    
    for idx, row in test_df.iterrows():
        # Создание тестового промпта
        prompt = f"""Текст сценария фильма "{row['title']}" ({row['year']}):

{row['text']}

Проанализируй сценарий и определи:
1. Рейтинг фильма от 1 до 10
2. Возрастной рейтинг (18+, 16+, 12+, 6+, 0+)

Ответ:"""
        
        # Генерация
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        
        with torch.no_grad():
            outputs = model.generate(
                inputs.input_ids.cuda(),
                max_new_tokens=50,
                temperature=0.3,
                do_sample=False
            )
        
        answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
        generated_text = answer[len(prompt):].strip()
        
        # Извлечение предсказаний
        pred_rating = None
        pred_age = None
        
        # Поиск числового рейтинга
        numbers = re.findall(r'(\d+\.?\d*)', generated_text)
        if numbers:
            try:
                pred_rating = float(numbers[0])
            except:
                pass
        
        # Поиск возрастного рейтинга
        age_patterns = [r'18\+', r'16\+', r'12\+', r'6\+', r'0\+']
        for pattern in age_patterns:
            if re.search(pattern, generated_text):
                pred_age = re.search(pattern, generated_text).group()
                break
        
        # Сравнение с фактическими значениями
        actual_rating = row['kp_rating']
        actual_age = row['age_rating']
        
        if pred_rating and abs(pred_rating - actual_rating) <= 1.0:
            correct_rating += 1
        
        if pred_age and pred_age == actual_age:
            correct_age += 1
        
        predictions.append({
            'title': row['title'],
            'actual_rating': actual_rating,
            'predicted_rating': pred_rating,
            'actual_age': actual_age,
            'predicted_age': pred_age,
            'response': generated_text
        })
    
    # Расчет метрик
    rating_accuracy = correct_rating / total * 100
    age_accuracy = correct_age / total * 100
    
    print(f"\n{'='*50}")
    print("ОЦЕНКА КАЧЕСТВА МОДЕЛИ")
    print(f"{'='*50}")
    print(f"Точность предсказания рейтинга (±1 балл): {rating_accuracy:.1f}%")
    print(f"Точность предсказания возрастного рейтинга: {age_accuracy:.1f}%")
    
    # Сохранение результатов
    results_df = pd.DataFrame(predictions)
    results_path = OUTPUT_DIR / "predictions_results.csv"
    results_df.to_csv(results_path, index=False, encoding=ENCODING)
    print(f"\nРезультаты сохранены в: {results_path}")
    
    # Вывод примеров
    print(f"\nПримеры предсказаний:")
    for i in range(min(3, len(results_df))):
        print(f"\n{i+1}. {results_df.iloc[i]['title']}")
        print(f"   Фактический рейтинг: {results_df.iloc[i]['actual_rating']:.1f}")
        print(f"   Предсказанный рейтинг: {results_df.iloc[i]['predicted_rating']}")
        print(f"   Фактический возрастной рейтинг: {results_df.iloc[i]['actual_age']}")
        print(f"   Предсказанный возрастной рейтинг: {results_df.iloc[i]['predicted_age']}")

# Оценка модели на валидационных данных
print("\n" + "="*50)
print("ЗАПУСК ОЦЕНКИ МОДЕЛИ")
print("="*50)

# Для оценки используем небольшую часть валидационных данных
eval_df = val_df.head(5).copy()
evaluate_model(model, tokenizer, eval_df)

# ==================================================
# ФИНАЛЬНЫЙ ЭКСПОРТ И СОХРАНЕНИЕ
# ==================================================
print("\n" + "="*50)
print("ФИНАЛЬНЫЙ ЭКСПОРТ")
print("="*50)

# Сохранение конфигурации
config = {
    "model_name": MODEL_NAME,
    "training_samples": len(train_df),
    "validation_samples": len(val_df),
    "min_text_length": MIN_TEXT_LENGTH,
    "max_text_length": MAX_TEXT_LENGTH,
    "date_trained": pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')
}

import json
with open(OUTPUT_DIR / "training_config.json", "w", encoding=ENCODING) as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("✓ Конфигурация сохранена")
print("✓ Модель готова к использованию")
print(f"\nДля тестирования на новых файлах используйте функцию:")
print(f"predict_movie_ratings('путь/к/файлу.txt', model, tokenizer)")
print(f"\nИли запустите интерактивный режим: interactive_testing()")

# YEs

In [1]:
# ==================================================
# ИНИЦИАЛИЗАЦИЯ И ПРОВЕРКА CUDA
# ==================================================
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

# ==================================================
# НАСТРОЙКА ПУТЕЙ И КОНФИГУРАЦИЯ
# ==================================================
from pathlib import Path
import pandas as pd
import re
import os
import warnings
from typing import Dict, List, Optional
from datasets import Dataset
# Конфигурация путей
ROOT = Path(".").resolve() 
OUTPUT_DIR = ROOT / "Out"
DATA_DIR = ROOT / "datasets"
METADATA_CSV = DATA_DIR / "scripts_ratingss.csv"
PROCESSED_DIR = DATA_DIR / "Pipe"
RAW_DATA_DIR = DATA_DIR / "scenaryy"
TEST_TXT_DIR = DATA_DIR / "noinfo"
MORE_TEST_TXT_DIR = DATA_DIR / "wrong"

# Параметры
MIN_TEXT_LENGTH = 1000
MAX_TEXT_LENGTH = 260000
ENCODING = 'utf-8'

# Создание директорий
for dir_path in [RAW_DATA_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print("Конфигурация загружена!")
print(f"ROOT: {ROOT}")

# ==================================================
# ЗАГРУЗКА И ПРЕДОБРАБОТКА ДАННЫХ
# ==================================================
class DataLoader:
    @staticmethod
    def load_metadata(csv_path: Path) -> pd.DataFrame:
        try:
            df = pd.read_csv(csv_path, encoding=ENCODING)
            print(f"Загружено {len(df)} записей из {csv_path.name}")
            return df
        except Exception as e:
            raise Exception(f"Ошибка загрузки CSV: {e}")
    
    @staticmethod
    def load_script_txt(file_path: Path) -> str:
        try:
            with open(file_path, 'r', encoding=ENCODING) as f:
                text = f.read()
            return text
        except UnicodeDecodeError:
            for encoding in ['cp1251', 'iso-8859-1', 'mac_cyrillic']:
                try:
                    with open(file_path, 'r', encoding=encoding) as f:
                        return f.read()
                except:
                    continue
            raise Exception(f"Не удалось декодировать файл: {file_path.name}")

class TextPreprocessor:
    @staticmethod
    def clean_script_text(raw_text: str) -> str:
        text = raw_text
        text = re.sub(r'\r\n', '\n', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[ \t]{2,}', ' ', text)
        text = re.sub(r'\([^)]*\)', '', text)
        text = re.sub(r'^[A-ZА-Я\s\d\-\.]+$', '', text, flags=re.MULTILINE)
        text = re.sub(r'[^\w\s\.,!?;:\-\'\"\(\)\n]', '', text)
        text = re.sub(r'\s+([.,!?;:])', r'\1', text)
        text = re.sub(r'([.,!?;:])\s+', r'\1 ', text)
        return text.strip()

# Загрузка метаданных
metadata_df = DataLoader.load_metadata(METADATA_CSV)
print("\nПервые 3 записи:")
print(metadata_df.head(3))

# ==================================================
# ПОДГОТОВКА ДАТАСЕТА ДЛЯ ОБУЧЕНИЯ
# ==================================================
def prepare_training_data(metadata_df, raw_data_dir, processed_dir):
    processed_data = []
    
    for idx, row in metadata_df.iterrows():
        try:
            filename = str(row['filename'])
            if not filename.endswith('.txt'):
                filename += '.txt'
            
            txt_path = raw_data_dir / filename
            
            if not txt_path.exists():
                print(f"Файл не найден: {filename}")
                continue
            
            # Загрузка и очистка текста
            raw_text = DataLoader.load_script_txt(txt_path)
            cleaned_text = TextPreprocessor.clean_script_text(raw_text)
            
            if len(cleaned_text) < MIN_TEXT_LENGTH:
                continue
            
            # Создание записи для обучения
            record = {
                'filename': filename,
                'title': row['title'],
                'year': row['year'],
                'kp_rating': row['kp_rating'],
                'imdb_rating': row['imdb_rating'],
                'age_rating': row['age_rating_kp'] if pd.notna(row['age_rating_kp']) else row['age_rating_imdb'],
                'text': cleaned_text[:5000]  # Берем первые 5000 символов для обучения
            }
            processed_data.append(record)
            
        except Exception as e:
            print(f"Ошибка обработки файла {filename}: {e}")
            continue
    
    return pd.DataFrame(processed_data)

# Создание датасета
processed_df = prepare_training_data(metadata_df, RAW_DATA_DIR, PROCESSED_DIR)
print(f"\nСоздан датасет: {len(processed_df)} записей")

# ==================================================
# РАЗДЕЛЕНИЕ НА ТРЕНИРОВОЧНУЮ И ТЕСТОВУЮ ВЫБОРКИ
# ==================================================
from sklearn.model_selection import train_test_split

# Создание бинарных меток для рейтинга (выше/ниже медианы)
median_rating = processed_df['kp_rating'].median()
processed_df['rating_label'] = (processed_df['kp_rating'] > median_rating).astype(int)

# Преобразование возрастного рейтинга в числовые метки
age_mapping = {
    '18+': 0,
    '16+': 1,
    '12+': 2,
    '6+': 3,
    '0+': 4,
    'Not found': 5
}
processed_df['age_label'] = processed_df['age_rating'].map(age_mapping).fillna(5).astype(int)

# Разделение данных
train_df, val_df = train_test_split(
    processed_df,
    test_size=0.2,
    random_state=42,
    stratify=processed_df['rating_label']
)

print(f"\nРазделение данных:")
print(f"Тренировочная выборка: {len(train_df)} записей")
print(f"Валидационная выборка: {len(val_df)} записей")

# ==================================================
# СОЗДАНИЕ ПРОМПТОВ ДЛЯ ОБУЧЕНИЯ
# ==================================================
def create_prompts(df, task_type="rating"):
    """Создание промптов для разных задач"""
    prompts = []
    
    for idx, row in df.iterrows():
        if task_type == "rating":
            # Промпт для предсказания рейтинга
            prompt = f"""Текст сценария фильма "{row['title']}" ({row['year']}):

{row['text']}

На основе этого текста сценария, оцени рейтинг фильма от 1 до 10.
Рейтинг:"""
            answer = f" {row['kp_rating']:.1f}"
            
        elif task_type == "age_rating":
            # Промпт для предсказания возрастного рейтинга
            prompt = f"""Текст сценария фильма "{row['title']}" ({row['year']}):

{row['text']}

Определи возрастной рейтинг для этого фильма (18+, 16+, 12+, 6+, 0+).
Возрастной рейтинг:"""
            answer = f" {row['age_rating']}"
        
        elif task_type == "both":
            # Промпт для предсказания обоих показателей
            prompt = f"""Текст сценария фильма "{row['title']}" ({row['year']}):

{row['text']}

Проанализируй сценарий и определи:
1. Рейтинг фильма от 1 до 10
2. Возрастной рейтинг (18+, 16+, 12+, 6+, 0+)

Ответ:"""
            answer = f" Рейтинг: {row['kp_rating']:.1f}, Возрастной рейтинг: {row['age_rating']}"
        
        prompts.append(prompt + answer)
    
    return prompts

# Создание промптов для разных задач
train_prompts_rating = create_prompts(train_df, "rating")
val_prompts_rating = create_prompts(val_df, "rating")

train_prompts_age = create_prompts(train_df, "age_rating")
val_prompts_age = create_prompts(val_df, "age_rating")

train_prompts_both = create_prompts(train_df, "both")
val_prompts_both = create_prompts(val_df, "both")

# Сохранение промптов
prompts_dir = PROCESSED_DIR / "prompts"
prompts_dir.mkdir(exist_ok=True)

with open(prompts_dir / "train_rating.txt", "w", encoding=ENCODING) as f:
    f.write("\n\n".join(train_prompts_rating))

with open(prompts_dir / "val_rating.txt", "w", encoding=ENCODING) as f:
    f.write("\n\n".join(val_prompts_rating))

print(f"\nПромпты сохранены в: {prompts_dir}")

# ==================================================
# УСТАНОВКА БИБЛИОТЕК И ИНИЦИАЛИЗАЦИЯ МОДЕЛИ
# ==================================================
print("\n" + "="*50)
print("УСТАНОВКА И ЗАГРУЗКА МОДЕЛИ")
print("="*50)

# Установка библиотек (если нужно)
import sys
import subprocess
import importlib



# Импорт библиотек
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# ==================================================
# КОНФИГУРАЦИЯ МОДЕЛИ
# ==================================================
MODEL_NAME = "Qwen/Qwen2.5-1.5B"

# Конфигурация квантизации для экономии памяти
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Загрузка токенизатора
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    trust_remote_code=True
)

# Добавление pad token если его нет
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ==================================================
# ПОДГОТОВКА ДАННЫХ ДЛЯ ОБУЧЕНИЯ
# ==================================================
def prepare_dataset_for_finetuning(prompts_list):
    """Подготовка датасета для тонкой настройки"""
    dataset = Dataset.from_dict({"text": prompts_list})
    
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )
    
    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=["text"]
    )
    
    return tokenized_dataset

# Подготовка датасетов
train_dataset = prepare_dataset_for_finetuning(train_prompts_both)
val_dataset = prepare_dataset_for_finetuning(val_prompts_both)

print(f"\nРазмеры датасетов:")
print(f"Обучающий: {len(train_dataset)} примеров")
print(f"Валидационный: {len(val_dataset)} примеров")

# ==================================================
# НАСТРОЙКА И ОБУЧЕНИЕ МОДЕЛИ С LoRA
# ==================================================
# Загрузка модели
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Подготовка модели для k-bit обучения
model = prepare_model_for_kbit_training(model)

# Конфигурация LoRA
lora_config = LoraConfig(
    r=8,  # Rank
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

# Применение LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Параметры обучения
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR / "model_output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=True,
    optim="paged_adamw_8bit",
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none"
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("\n" + "="*50)
print("НАЧАЛО ОБУЧЕНИЯ")
print("="*50)

# Обучение модели
trainer.train()

# Сохранение модели
model.save_pretrained(OUTPUT_DIR / "trained_model")
tokenizer.save_pretrained(OUTPUT_DIR / "trained_model")
print(f"\nМодель сохранена в: {OUTPUT_DIR / 'trained_model'}")

# ==================================================
# ФУНКЦИЯ ДЛЯ ПРЕДСКАЗАНИЯ НА НОВЫХ ФАЙЛАХ
# ==================================================
def predict_movie_ratings(text_file_path, model, tokenizer):
    """
    Предсказание рейтинга и возрастного рейтинга для нового сценария
    
    Args:
        text_file_path: Путь к текстовому файлу со сценарием
        model: Обученная модель
        tokenizer: Токенизатор
    """
    # Загрузка текста
    with open(text_file_path, 'r', encoding='utf-8') as f:
        script_text = f.read()
    
    # Очистка текста
    cleaned_text = TextPreprocessor.clean_script_text(script_text)
    
    # Создание промпта
    prompt = f"""Текст сценария фильма:

{cleaned_text[:3000]}

Проанализируй сценарий и определи:
1. Рейтинг фильма от 1 до 10
2. Возрастной рейтинг (18+, 16+, 12+, 6+, 0+)

Ответ:"""
    
    # Токенизация
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    
    # Генерация ответа
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids.cuda(),
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            top_p=0.9
        )
    
    # Декодирование ответа
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Извлечение ответа (только часть после промпта)
    generated_text = answer[len(prompt):].strip()
    
    # Парсинг ответа
    rating = None
    age_rating = None
    
    # Поиск рейтинга
    rating_match = re.search(r'Рейтинг:\s*([0-9.]+)', generated_text)
    if rating_match:
        rating = float(rating_match.group(1))
    
    # Поиск возрастного рейтинга
    age_match = re.search(r'Возрастной рейтинг:\s*([0-9+]+)', generated_text)
    if not age_match:
        age_match = re.search(r'([0-9]+\+)', generated_text)
    
    if age_match:
        age_rating = age_match.group(1)
    
    return {
        "filename": text_file_path.name,
        "predicted_rating": rating,
        "predicted_age_rating": age_rating,
        "full_response": generated_text
    }

# ==================================================
# ТЕСТИРОВАНИЕ НА ТЕСТОВЫХ ФАЙЛАХ
# ==================================================
print("\n" + "="*50)
print("ТЕСТИРОВАНИЕ НА НОВЫХ ФАЙЛАХ")
print("="*50)

# Тестирование на файлах из тестовых директорий
test_dirs = [TEST_TXT_DIR, MORE_TEST_TXT_DIR]

for test_dir in test_dirs:
    if test_dir.exists():
        print(f"\nТестирование файлов из: {test_dir}")
        
        # Поиск всех txt файлов
        test_files = list(test_dir.glob("*.txt"))
        
        for test_file in test_files[:3]:  # Тестируем первые 3 файла
            try:
                result = predict_movie_ratings(test_file, model, tokenizer)
                print(f"\nФайл: {result['filename']}")
                print(f"Предсказанный рейтинг: {result['predicted_rating']}")
                print(f"Предсказанный возрастной рейтинг: {result['predicted_age_rating']}")
                print(f"Полный ответ: {result['full_response'][:100]}...")
            except Exception as e:
                print(f"Ошибка при обработке {test_file.name}: {e}")

# ==================================================
# ИНТЕРФЕЙС ДЛЯ РУЧНОГО ТЕСТИРОВАНИЯ
# ==================================================
def interactive_testing():
    """Интерактивный режим тестирования"""
    print("\n" + "="*50)
    print("ИНТЕРФЕКТИВНЫЙ РЕЖИМ ТЕСТИРОВАНИЯ")
    print("="*50)
    print("Введите путь к текстовому файлу со сценарием (или 'exit' для выхода)")
    
    while True:
        file_path = input("\nПуть к файлу: ").strip()
        
        if file_path.lower() == 'exit':
            break
        
        file_path = Path(file_path)
        
        if not file_path.exists():
            print(f"Файл не найден: {file_path}")
            continue
        
        try:
            result = predict_movie_ratings(file_path, model, tokenizer)
            print(f"\n{'='*40}")
            print(f"РЕЗУЛЬТАТЫ ДЛЯ: {result['filename']}")
            print(f"{'='*40}")
            print(f"📊 Предсказанный рейтинг: {result['predicted_rating']:.1f}/10")
            print(f"🎭 Возрастной рейтинг: {result['predicted_age_rating']}")
            print(f"\n📝 Анализ модели:")
            print(f"{result['full_response']}")
            print(f"{'='*40}")
            
        except Exception as e:
            print(f"Ошибка: {e}")

# Запуск интерактивного режима (раскомментировать при необходимости)
# interactive_testing()

# ==================================================
# ФИНАЛЬНЫЙ ЭКСПОРТ И СОХРАНЕНИЕ
# ==================================================
print("\n" + "="*50)
print("ФИНАЛЬНЫЙ ЭКСПОРТ")
print("="*50)

# Сохранение конфигурации
config = {
    "model_name": MODEL_NAME,
    "training_samples": len(train_df),
    "validation_samples": len(val_df),
    "min_text_length": MIN_TEXT_LENGTH,
    "max_text_length": MAX_TEXT_LENGTH,
    "date_trained": pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')
}

import json
with open(OUTPUT_DIR / "training_config.json", "w", encoding=ENCODING) as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("✓ Конфигурация сохранена")
print("✓ Модель готова к использованию")
print(f"\nДля тестирования на новых файлах используйте функцию:")
print(f"predict_movie_ratings('путь/к/файлу.txt', model, tokenizer)")
print(f"\nИли запустите интерактивный режим: interactive_testing()")

CUDA: True
NVIDIA GeForce RTX 5070 Ti


c:\Users\Дмитрий\Downloads\Ratings\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Конфигурация загружена!
ROOT: C:\Users\Дмитрий\Downloads\Ratings
Загружено 51 записей из scripts_ratingss.csv

Первые 3 записи:
                    filename                  title  year  kp_rating  \
0  13_причин_почему_Кино.txt  13_причин_почему_Кино  2017        7.3   
1    Игра_Престолов_Кино.txt    Игра_Престолов_Кино  2011        9.0   
2     8_миллиметров_Кино.txt     8_миллиметров_Кино  1999        7.1   

   imdb_rating notes age_rating_imdb age_rating_kp           english_title  
0          7.4    ok           TV-MA           16+    TH1RTEEN R3ASONS WHY  
1          9.2    ok           TV-MA           18+  A Song of Ice and Fire  
2          6.6    ok               R           18+            8 Millimeter  

Создан датасет: 51 записей

Разделение данных:
Тренировочная выборка: 40 записей
Валидационная выборка: 11 записей

Промпты сохранены в: C:\Users\Дмитрий\Downloads\Ratings\datasets\Pipe\prompts

УСТАНОВКА И ЗАГРУЗКА МОДЕЛИ


Map: 100%|██████████| 11/11 [00:00<00:00, 876.84 examples/s]



Размеры датасетов:
Обучающий: 40 примеров
Валидационный: 11 примеров
trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410

НАЧАЛО ОБУЧЕНИЯ


...

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



Файл: АУЕ_Кино.txt
Предсказанный рейтинг: None
Предсказанный возрастной рейтинг: None
Полный ответ: ...


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



Файл: Барри_Кино.txt
Предсказанный рейтинг: None
Предсказанный возрастной рейтинг: None
Полный ответ: ...


KeyboardInterrupt: 

# TRY 2

In [1]:
# ==================================================
# ИНИЦИАЛИЗАЦИЯ И ПРОВЕРКА CUDA
# ==================================================
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

# ==================================================
# НАСТРОЙКА ПУТЕЙ И КОНФИГУРАЦИЯ
# ==================================================
from pathlib import Path
import pandas as pd
import re
import os
import warnings
from typing import Dict, List, Optional

# Конфигурация путей
ROOT = Path(".").resolve() 
OUTPUT_DIR = ROOT / "Out"
DATA_DIR = ROOT / "datasets"
METADATA_CSV = DATA_DIR / "scripts_ratingss.csv"
PROCESSED_DIR = DATA_DIR / "Pipe"
RAW_DATA_DIR = DATA_DIR / "scenaryy"
TEST_TXT_DIR = DATA_DIR / "noinfo"

# Параметры
MIN_TEXT_LENGTH = 1000
MAX_TEXT_LENGTH = 100000
ENCODING = 'utf-8'

# Создание директорий
for dir_path in [RAW_DATA_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print("Конфигурация загружена!")
print(f"ROOT: {ROOT}")

# ==================================================
# ЗАГРУЗКА И ПРЕДОБРАБОТКА ДАННЫХ
# ==================================================
class DataLoader:
    @staticmethod
    def load_metadata(csv_path: Path) -> pd.DataFrame:
        try:
            df = pd.read_csv(csv_path, encoding=ENCODING)
            print(f"Загружено {len(df)} записей из {csv_path.name}")
            return df
        except Exception as e:
            raise Exception(f"Ошибка загрузки CSV: {e}")
    
    @staticmethod
    def load_script_txt(file_path: Path) -> str:
        try:
            with open(file_path, 'r', encoding=ENCODING) as f:
                text = f.read()
            return text
        except UnicodeDecodeError:
            for encoding in ['cp1251', 'iso-8859-1', 'mac_cyrillic']:
                try:
                    with open(file_path, 'r', encoding=encoding) as f:
                        return f.read()
                except:
                    continue
            raise Exception(f"Не удалось декодировать файл: {file_path.name}")

class TextPreprocessor:
    @staticmethod
    def clean_script_text(raw_text: str) -> str:
        text = raw_text
        text = re.sub(r'\r\n', '\n', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[ \t]{2,}', ' ', text)
        text = re.sub(r'\([^)]*\)', '', text)
        text = re.sub(r'^[A-ZА-Я\s\d\-\.]+$', '', text, flags=re.MULTILINE)
        text = re.sub(r'[^\w\s\.,!?;:\-\'\"\(\)\n]', '', text)
        text = re.sub(r'\s+([.,!?;:])', r'\1', text)
        text = re.sub(r'([.,!?;:])\s+', r'\1 ', text)
        return text.strip()

# Загрузка метаданных
metadata_df = DataLoader.load_metadata(METADATA_CSV)
print("\nПервые 3 записи:")
print(metadata_df.head(3))

# ==================================================
# ПОДГОТОВКА ДАТАСЕТА ДЛЯ ОБУЧЕНИЯ (ПОЛНЫЙ СЦЕНАРИЙ!)
# ==================================================
def prepare_training_data(metadata_df, raw_data_dir, processed_dir):
    processed_data = []
    
    for idx, row in metadata_df.iterrows():
        try:
            filename = str(row['filename'])
            if not filename.endswith('.txt'):
                filename += '.txt'
            
            txt_path = raw_data_dir / filename
            
            if not txt_path.exists():
                print(f"Файл не найден: {filename}")
                continue
            
            # Загрузка и очистка текста (ВЕСЬ текст!)
            raw_text = DataLoader.load_script_txt(txt_path)
            cleaned_text = TextPreprocessor.clean_script_text(raw_text)
            
            if len(cleaned_text) < MIN_TEXT_LENGTH:
                print(f"Текст слишком короткий: {filename} ({len(cleaned_text)} символов)")
                continue
            
            # Используем ВЕСЬ текст для обучения
            # Но обрежем до максимальной длины, если нужно
            if len(cleaned_text) > MAX_TEXT_LENGTH:
                cleaned_text = cleaned_text[:MAX_TEXT_LENGTH]
                print(f"Обрезан длинный текст: {filename} ({len(cleaned_text)} символов)")
            
            # Создание записи для обучения
            record = {
                'filename': filename,
                'title': row['title'],
                'year': row['year'],
                'kp_rating': row['kp_rating'],
                'imdb_rating': row['imdb_rating'],
                'age_rating': row['age_rating_kp'] if pd.notna(row['age_rating_kp']) else row['age_rating_imdb'],
                'text': cleaned_text  # ВЕСЬ текст!
            }
            processed_data.append(record)
            
        except Exception as e:
            print(f"Ошибка обработки файла {filename}: {e}")
            continue
    
    return pd.DataFrame(processed_data)

# Создание датасета
processed_df = prepare_training_data(metadata_df, RAW_DATA_DIR, PROCESSED_DIR)
print(f"\nСоздан датасет: {len(processed_df)} записей")
print(f"Средняя длина текста: {processed_df['text'].apply(len).mean():.0f} символов")

# ==================================================
# РАЗДЕЛЕНИЕ НА ТРЕНИРОВОЧНУЮ И ТЕСТОВУЮ ВЫБОРКИ
# ==================================================
from sklearn.model_selection import train_test_split

# Создание бинарных меток для рейтинга (выше/ниже медианы)
median_rating = processed_df['kp_rating'].median()
processed_df['rating_label'] = (processed_df['kp_rating'] > median_rating).astype(int)

# Преобразование возрастного рейтинга в числовые метки
age_mapping = {
    '18+': 0,
    '16+': 1,
    '12+': 2,
    '6+': 3,
    '0+': 4,
    'Not found': 5,
    'NOT_RATED': 5,
    'NR': 5,
    'R': 0  # R примерно соответствует 18+
}
processed_df['age_label'] = processed_df['age_rating'].map(age_mapping).fillna(5).astype(int)

# Разделение данных
train_df, val_df = train_test_split(
    processed_df,
    test_size=0.15,  # Уменьшил для большего тренировочного набора
    random_state=42,
    stratify=processed_df['rating_label']
)

print(f"\nРазделение данных:")
print(f"Тренировочная выборка: {len(train_df)} записей")
print(f"Валидационная выборка: {len(val_df)} записей")

# ==================================================
# УЛУЧШЕННОЕ СОЗДАНИЕ ПРОМПТОВ ДЛЯ ОБУЧЕНИЯ
# ==================================================
def create_prompts(df, task_type="both"):
    """Создание улучшенных промптов для обучения"""
    prompts = []
    
    for idx, row in df.iterrows():
        # Берем первые 15000 символов для промпта (можно увеличить)
        text_for_prompt = row['text'][:15000]
        
        if task_type == "both":
            # УЛУЧШЕННЫЙ промпт для предсказания обоих показателей
            prompt = f"""ТЕКСТ СЦЕНАРИЯ ФИЛЬМА:

{text_for_prompt}

ПРОАНАЛИЗИРУЙ ЭТОТ СЦЕНАРИЙ И ДАЙ ОЦЕНКУ:

РЕЙТИНГ (от 1.0 до 10.0, где 10.0 - отлично):
ВОЗРАСТНОЙ РЕЙТИНГ (выбери один: 0+, 6+, 12+, 16+, 18+):

ОТВЕТ:
РЕЙТИНГ = {row['kp_rating']:.1f}
ВОЗРАСТНОЙ РЕЙТИНГ = {row['age_rating']}"""
        
        elif task_type == "rating":
            # УЛУЧШЕННЫЙ промпт для предсказания рейтинга
            prompt = f"""ТЕКСТ СЦЕНАРИЯ ФИЛЬМА:

{text_for_prompt}

НА ОСНОВЕ ЭТОГО ТЕКСТА СЦЕНАРИЯ, ОЦЕНИ РЕЙТИНГ ФИЛЬМА ОТ 1.0 ДО 10.0 (ГДЕ 10.0 - ОТЛИЧНО):

ОТВЕТ: {row['kp_rating']:.1f}"""
            
        elif task_type == "age_rating":
            # УЛУЧШЕННЫЙ промпт для предсказания возрастного рейтинга
            prompt = f"""ТЕКСТ СЦЕНАРИЯ ФИЛЬМА:

{text_for_prompt}

ОПРЕДЕЛИ ВОЗРАСТНОЙ РЕЙТИНГ ДЛЯ ЭТОГО ФИЛЬМА (ВЫБЕРИ ОДИН ИЗ: 0+, 6+, 12+, 16+, 18+):

ОТВЕТ: {row['age_rating']}"""
        
        prompts.append(prompt)
    
    return prompts

# Создание УЛУЧШЕННЫХ промптов
train_prompts = create_prompts(train_df, "both")
val_prompts = create_prompts(val_df, "both")

# Сохранение промптов
prompts_dir = PROCESSED_DIR / "prompts"
prompts_dir.mkdir(exist_ok=True)

with open(prompts_dir / "train_prompts.txt", "w", encoding=ENCODING) as f:
    f.write("\n\n".join(train_prompts))

with open(prompts_dir / "val_prompts.txt", "w", encoding=ENCODING) as f:
    f.write("\n\n".join(val_prompts))

print(f"\nПромпты сохранены в: {prompts_dir}")

# ==================================================
# УСТАНОВКА БИБЛИОТЕК И ИНИЦИАЛИЗАЦИЯ МОДЕЛИ
# ==================================================
print("\n" + "="*50)
print("УСТАНОВКА И ЗАГРУЗКА МОДЕЛИ")
print("="*50)

# Импорт библиотек
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import torch

# ==================================================
# КОНФИГУРАЦИЯ МОДЕЛИ
# ==================================================
MODEL_NAME = "Qwen/Qwen2.5-1.5B"

# Конфигурация квантизации для экономии памяти
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Загрузка токенизатора
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    trust_remote_code=True
)

# Настройка токенизатора
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = 'right'

# ==================================================
# ПОДГОТОВКА ДАННЫХ ДЛЯ ОБУЧЕНИЯ
# ==================================================
def prepare_dataset_for_finetuning(prompts_list, max_length=2048):
    """Подготовка датасета для тонкой настройки"""
    dataset = Dataset.from_dict({"text": prompts_list})
    
    def tokenize_function(examples):
        # Токенизация с вниманием к длине
        result = tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt"
        )
        
        # Создаем labels (такие же как input_ids)
        result["labels"] = result["input_ids"].clone()
        
        return result
    
    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=["text"]
    )
    
    return tokenized_dataset

# Подготовка датасетов с большей длиной
print("\nПодготовка датасетов...")
train_dataset = prepare_dataset_for_finetuning(train_prompts, max_length=2048)
val_dataset = prepare_dataset_for_finetuning(val_prompts, max_length=2048)

print(f"Размеры датасетов:")
print(f"Обучающий: {len(train_dataset)} примеров")
print(f"Валидационный: {len(val_dataset)} примеров")

# ==================================================
# НАСТРОЙКА И ОБУЧЕНИЕ МОДЕЛИ С LoRA
# ==================================================
print("\n" + "="*50)
print("НАСТРОЙКА МОДЕЛИ")
print("="*50)

# Загрузка модели
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Подготовка модели для k-bit обучения
model = prepare_model_for_kbit_training(model)

# УЛУЧШЕННАЯ конфигурация LoRA
lora_config = LoraConfig(
    r=16,  # Увеличил rank для лучшего обучения
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,  # Уменьшил dropout
    bias="none",
    task_type="CAUSAL_LM"
)

# Применение LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# УЛУЧШЕННЫЕ параметры обучения
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR / "model_output",
    num_train_epochs=5,  # Увеличил количество эпох
    per_device_train_batch_size=1,  # Уменьшил batch size для работы с длинными текстами
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=100,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    learning_rate=1e-4,  # Уменьшил learning rate
    fp16=True,
    optim="paged_adamw_8bit",
    save_total_limit=3,
    load_best_model_at_end=True,
    report_to="none",
    gradient_checkpointing=True,  # Включил для экономии памяти
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("\n" + "="*50)
print("НАЧАЛО ОБУЧЕНИЯ")
print("="*50)

# Обучение модели
trainer.train()

# Сохранение модели
model.save_pretrained(OUTPUT_DIR / "trained_model")
tokenizer.save_pretrained(OUTPUT_DIR / "trained_model")
print(f"\n✅ Модель сохранена в: {OUTPUT_DIR / 'trained_model'}")

# ==================================================
# УЛУЧШЕННАЯ ФУНКЦИЯ ДЛЯ ПРЕДСКАЗАНИЯ
# ==================================================
def predict_movie_ratings_improved(text_file_path, model, tokenizer, max_chars=20000):
    """
    УЛУЧШЕННОЕ предсказание рейтинга и возрастного рейтинга
    
    Args:
        text_file_path: Путь к текстовому файлу со сценарием
        model: Обученная модель
        tokenizer: Токенизатор
        max_chars: Максимальное количество символов для анализа
    """
    # Загрузка текста
    with open(text_file_path, 'r', encoding='utf-8') as f:
        script_text = f.read()
    
    # Очистка текста
    cleaned_text = TextPreprocessor.clean_script_text(script_text)
    
    # Обрезаем текст если слишком длинный
    if len(cleaned_text) > max_chars:
        cleaned_text = cleaned_text[:max_chars]
    
    # УЛУЧШЕННЫЙ промпт
    prompt = f"""ТЕКСТ СЦЕНАРИЯ ФИЛЬМА:

{cleaned_text}

ПРОАНАЛИЗИРУЙ ЭТОТ СЦЕНАРИЙ И ДАЙ ОЦЕНКУ:

РЕЙТИНГ (от 1.0 до 10.0, где 10.0 - отлично):
ВОЗРАСТНОЙ РЕЙТИНГ (выбери один: 0+, 6+, 12+, 16+, 18+):

ОТВЕТ:
РЕЙТИНГ ="""
    
    # Токенизация с вниманием к деталям
    inputs = tokenizer(
        prompt, 
        return_tensors="pt", 
        truncation=True, 
        max_length=2048,
        padding=True
    )
    
    # Перемещаем на GPU если доступно
    device = model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # УЛУЧШЕННАЯ генерация с лучшими параметрами
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.3,  # Низкая температура для более детерминированных ответов
            do_sample=False,   # Отключил sampling для более точных ответов
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # Декодирование ответа
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Извлечение ответа (только часть после промпта)
    generated_text = answer[len(prompt):].strip()
    
    # УЛУЧШЕННЫЙ парсинг ответа
    rating = None
    age_rating = None
    
    # Поиск рейтинга в формате "РЕЙТИНГ = X.X"
    rating_patterns = [
        r'РЕЙТИНГ\s*=\s*([0-9]\.[0-9]|[0-9])',
        r'Рейтинг:\s*([0-9]\.[0-9]|[0-9])',
        r'([0-9]\.[0-9])/10',
        r'([0-9]\.[0-9])'
    ]
    
    for pattern in rating_patterns:
        match = re.search(pattern, generated_text, re.IGNORECASE)
        if match:
            try:
                rating = float(match.group(1))
                # Ограничиваем диапазон 1.0-10.0
                if rating < 1.0:
                    rating = 1.0
                elif rating > 10.0:
                    rating = 10.0
                break
            except:
                continue
    
    # Поиск возрастного рейтинга
    age_patterns = [
        r'ВОЗРАСТНОЙ РЕЙТИНГ\s*=\s*([0-9]+\+)',
        r'Возрастной рейтинг:\s*([0-9]+\+)',
        r'([0-9]+\+)\s*возраст',
        r'([0-9]+\+)'
    ]
    
    for pattern in age_patterns:
        match = re.search(pattern, generated_text, re.IGNORECASE)
        if match:
            age_rating = match.group(1)
            # Нормализация
            if age_rating in ['18+', '16+', '12+', '6+', '0+']:
                break
            else:
                # Если не стандартный, пробуем преобразовать
                if '18' in age_rating:
                    age_rating = '18+'
                elif '16' in age_rating:
                    age_rating = '16+'
                elif '12' in age_rating:
                    age_rating = '12+'
                elif '6' in age_rating:
                    age_rating = '6+'
                elif '0' in age_rating:
                    age_rating = '0+'
                break
    
    return {
        "filename": text_file_path.name,
        "predicted_rating": rating,
        "predicted_age_rating": age_rating,
        "full_response": generated_text,
        "prompt_used": prompt[:200] + "..." if len(prompt) > 200 else prompt
    }

# ==================================================
# ТЕСТИРОВАНИЕ НА ТЕСТОВЫХ ФАЙЛАХ (ТОЛЬКО NOINFO!)
# ==================================================
print("\n" + "="*50)
print("ТЕСТИРОВАНИЕ НА НОВЫХ ФАЙЛАХ (ТОЛЬКО NOINFO)")
print("="*50)

# Только папка noinfo
test_dirs = [TEST_TXT_DIR]

all_test_results = []

for test_dir in test_dirs:
    if test_dir.exists():
        print(f"\n📂 Тестирование файлов из: {test_dir}")
        
        # Поиск всех txt файлов
        test_files = list(test_dir.glob("*.txt"))
        
        print(f"Найдено файлов: {len(test_files)}")
        
        for i, test_file in enumerate(test_files, 1):
            try:
                print(f"\n┌─ Файл {i}/{len(test_files)}: {test_file.name}")
                result = predict_movie_ratings_improved(test_file, model, tokenizer, max_chars=15000)
                all_test_results.append(result)
                
                print(f"├─ Предсказанный рейтинг: ", end="")
                if result['predicted_rating']:
                    print(f"✅ {result['predicted_rating']:.1f}/10")
                else:
                    print(f"❌ Не определен")
                
                print(f"├─ Возрастной рейтинг: ", end="")
                if result['predicted_age_rating']:
                    print(f"✅ {result['predicted_age_rating']}")
                else:
                    print(f"❌ Не определен")
                
                print(f"└─ Ответ модели: {result['full_response'][:80]}...")
                
            except Exception as e:
                print(f"❌ Ошибка при обработке {test_file.name}: {e}")

# ==================================================
# АНАЛИЗ РЕЗУЛЬТАТОВ ТЕСТИРОВАНИЯ
# ==================================================
print("\n" + "="*50)
print("АНАЛИЗ РЕЗУЛЬТАТОВ ТЕСТИРОВАНИЯ")
print("="*50)

if all_test_results:
    # Создаем DataFrame для анализа
    results_df = pd.DataFrame(all_test_results)
    
    # Статистика
    successful_rating = results_df['predicted_rating'].notna().sum()
    successful_age = results_df['predicted_age_rating'].notna().sum()
    total_tests = len(results_df)
    
    print(f"\n📊 СТАТИСТИКА:")
    print(f"Всего протестировано файлов: {total_tests}")
    print(f"Успешно определено рейтингов: {successful_rating}/{total_tests} ({successful_rating/total_tests*100:.1f}%)")
    print(f"Успешно определено возрастных рейтингов: {successful_age}/{total_tests} ({successful_age/total_tests*100:.1f}%)")
    
    # Распределение предсказанных рейтингов
    if successful_rating > 0:
        ratings = results_df['predicted_rating'].dropna()
        print(f"\n📈 Распределение предсказанных рейтингов:")
        print(f"Средний: {ratings.mean():.1f}")
        print(f"Медиана: {ratings.median():.1f}")
        print(f"Минимальный: {ratings.min():.1f}")
        print(f"Максимальный: {ratings.max():.1f}")
    
    # Распределение возрастных рейтингов
    if successful_age > 0:
        age_counts = results_df['predicted_age_rating'].value_counts()
        print(f"\n👥 Распределение возрастных рейтингов:")
        for age, count in age_counts.items():
            print(f"  {age}: {count} файлов")
    
    # Сохранение результатов
    results_path = OUTPUT_DIR / "test_results.csv"
    results_df.to_csv(results_path, index=False, encoding=ENCODING)
    print(f"\n💾 Результаты сохранены в: {results_path}")
    
    # Вывод лучших и худших предсказаний
    print(f"\n🏆 ТОП-3 результата:")
    for i, (idx, row) in enumerate(results_df.iterrows()):
        if i >= 3:
            break
        print(f"\n{i+1}. {row['filename']}")
        print(f"   Рейтинг: {row['predicted_rating'] if row['predicted_rating'] else 'Нет'}")
        print(f"   Возрастной: {row['predicted_age_rating'] if row['predicted_age_rating'] else 'Нет'}")
else:
    print("❌ Нет результатов для анализа")

# ==================================================
# ИНТЕРФЕЙС ДЛЯ РУЧНОГО ТЕСТИРОВАНИЯ
# ==================================================
def interactive_testing_improved():
    """УЛУЧШЕННЫЙ интерактивный режим тестирования"""
    print("\n" + "="*50)
    print("🔧 ИНТЕРАКТИВНЫЙ РЕЖИМ ТЕСТИРОВАНИЯ")
    print("="*50)
    print("Введите путь к текстовому файлу со сценарием")
    print("Или введите 'exit' для выхода")
    print("Или 'demo' для тестового примера")
    
    while True:
        user_input = input("\n📁 Путь к файлу: ").strip()
        
        if user_input.lower() == 'exit':
            break
        elif user_input.lower() == 'demo':
            # Создаем демо-файл
            demo_text = """ИНТРО - НОЧЬ
Камера медленно приближается к окну квартиры. За окном - ночной город.
ВНУТРИ КВАРТИРЫ
АЛЕКС (30), программист, сидит за компьютером. На экране - код.
Он выглядит уставшим, но сосредоточенным.
АЛЕКС
(шепотом)
Еще немного... почти готово...
Он печатает последние строки кода, затем откидывается на спинку кресла.
АЛЕКС
(улыбаясь)
Сделано.
ФИНАЛЬНЫЕ ТИТРЫ"""
            
            demo_path = OUTPUT_DIR / "demo_script.txt"
            with open(demo_path, 'w', encoding='utf-8') as f:
                f.write(demo_text)
            
            print(f"\n📝 Создан демо-файл: {demo_path}")
            user_input = str(demo_path)
        
        file_path = Path(user_input)
        
        if not file_path.exists():
            print(f"❌ Файл не найден: {file_path}")
            continue
        
        try:
            print("\n" + "▬" * 50)
            print(f"🎬 АНАЛИЗ ФАЙЛА: {file_path.name}")
            print("▬" * 50)
            
            result = predict_movie_ratings_improved(file_path, model, tokenizer, max_chars=20000)
            
            print(f"\n📊 РЕЗУЛЬТАТЫ:")
            print(f"├─ Файл: {result['filename']}")
            
            if result['predicted_rating']:
                # Создаем визуальную шкалу рейтинга
                rating = result['predicted_rating']
                stars = "★" * int(rating)
                if rating - int(rating) >= 0.5:
                    stars += "½"
                stars = stars.ljust(10, '☆')
                
                print(f"├─ Рейтинг: {rating:.1f}/10")
                print(f"│  {stars}")
                
                # Интерпретация рейтинга
                if rating >= 9.0:
                    rating_desc = "ШЕДЕВР! 🏆"
                elif rating >= 8.0:
                    rating_desc = "ОТЛИЧНО! 👍"
                elif rating >= 7.0:
                    rating_desc = "ХОРОШО 👌"
                elif rating >= 6.0:
                    rating_desc = "НЕПЛОХО 🙂"
                elif rating >= 5.0:
                    rating_desc = "СРЕДНЕ 😐"
                else:
                    rating_desc = "НИЖЕ СРЕДНЕГО 👎"
                
                print(f"│  {rating_desc}")
            else:
                print(f"├─ Рейтинг: ❌ Не определен")
            
            if result['predicted_age_rating']:
                age = result['predicted_age_rating']
                print(f"├─ Возрастной рейтинг: {age}")
                
                # Описание возрастного рейтинга
                age_descriptions = {
                    '0+': "Для всех возрастов 👶",
                    '6+': "Для детей от 6 лет 🧒",
                    '12+': "Для подростков от 12 лет 🧑",
                    '16+': "Для молодежи от 16 лет 🧑‍🎓",
                    '18+': "Только для взрослых 🔞"
                }
                
                if age in age_descriptions:
                    print(f"│  {age_descriptions[age]}")
            else:
                print(f"├─ Возрастной рейтинг: ❌ Не определен")
            
            print(f"\n📝 ПОЛНЫЙ ОТВЕТ МОДЕЛИ:")
            print("─" * 40)
            print(result['full_response'])
            print("─" * 40)
            
            print(f"\n💡 СОВЕТ: ", end="")
            if result['predicted_rating'] and result['predicted_age_rating']:
                if result['predicted_rating'] >= 8.0 and result['predicted_age_rating'] == '18+':
                    print("Этот сценарий выглядит перспективным для взрослой аудитории!")
                elif result['predicted_rating'] >= 7.0:
                    print("Сценарий имеет хороший потенциал. Рекомендуется доработка.")
                else:
                    print("Сценарий требует существенной доработки.")
            else:
                print("Не удалось полностью проанализировать сценарий.")
            
            print("▬" * 50)
            
            # Сохранение результата
            save_result = input("\n💾 Сохранить результат? (да/нет): ").lower()
            if save_result in ['да', 'д', 'yes', 'y']:
                timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
                save_path = OUTPUT_DIR / f"analysis_{file_path.stem}_{timestamp}.txt"
                
                with open(save_path, 'w', encoding='utf-8') as f:
                    f.write(f"АНАЛИЗ СЦЕНАРИЯ: {file_path.name}\n")
                    f.write(f"Дата анализа: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                    f.write("="*50 + "\n\n")
                    f.write(f"📊 РЕЙТИНГ: {result['predicted_rating'] if result['predicted_rating'] else 'Нет'}/10\n")
                    f.write(f"👥 ВОЗРАСТНОЙ РЕЙТИНГ: {result['predicted_age_rating'] if result['predicted_age_rating'] else 'Нет'}\n\n")
                    f.write("📝 ПОЛНЫЙ ОТВЕТ МОДЕЛИ:\n")
                    f.write("-"*40 + "\n")
                    f.write(result['full_response'] + "\n")
                    f.write("-"*40 + "\n")
                
                print(f"✅ Результат сохранен в: {save_path}")
            
        except Exception as e:
            print(f"❌ Ошибка: {e}")
            import traceback
            traceback.print_exc()

# ==================================================
# ФИНАЛЬНЫЙ ЭКСПОРТ И СОХРАНЕНИЕ
# ==================================================
print("\n" + "="*50)
print("✅ ОБУЧЕНИЕ ЗАВЕРШЕНО!")
print("="*50)

# Сохранение итоговой конфигурации
config = {
    "model_name": MODEL_NAME,
    "training_samples": len(train_df),
    "validation_samples": len(val_df),
    "min_text_length": MIN_TEXT_LENGTH,
    "max_text_length": MAX_TEXT_LENGTH,
    "avg_text_length": processed_df['text'].apply(len).mean(),
    "test_files_analyzed": len(all_test_results) if all_test_results else 0,
    "date_trained": pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    "training_notes": "Обучение на полных сценариях, улучшенные промпты, тестирование только на noinfo"
}

with open(OUTPUT_DIR / "training_config.json", "w", encoding=ENCODING) as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("📋 КОНФИГУРАЦИЯ:")
for key, value in config.items():
    print(f"  {key}: {value}")

print("\n🚀 МОДЕЛЬ ГОТОВА К ИСПОЛЬЗОВАНИЮ!")
print("\n📌 КОМАНДЫ ДЛЯ ИСПОЛЬЗОВАНИЯ:")
print("1. Для интерактивного тестирования: interactive_testing_improved()")
print("2. Для анализа конкретного файла:")
print("   result = predict_movie_ratings_improved('путь/к/файлу.txt', model, tokenizer)")
print("\n📂 РЕЗУЛЬТАТЫ СОХРАНЕНЫ В:")
print(f"   - Модель: {OUTPUT_DIR / 'trained_model'}")
print(f"   - Конфигурация: {OUTPUT_DIR / 'training_config.json'}")
print(f"   - Результаты тестов: {OUTPUT_DIR / 'test_results.csv' if all_test_results else 'Нет'}")

# Сохраняем пример использования в файл
usage_example = f"""
ИНСТРУКЦИЯ ПО ИСПОЛЬЗОВАНИЮ МОДЕЛИ
===================================

Модель обучена предсказывать рейтинг фильма (1.0-10.0) и возрастной рейтинг (0+, 6+, 12+, 16+, 18+)
по тексту сценария.

ИСПОЛЬЗОВАНИЕ:

1. Интерактивный режим:
   >>> interactive_testing_improved()
   
2. Программный анализ:
   >>> from transformers import AutoTokenizer, AutoModelForCausalLM
   >>> from peft import PeftModel
   >>> import torch
   
   # Загрузка модели
   >>> tokenizer = AutoTokenizer.from_pretrained("{OUTPUT_DIR / 'trained_model'}")
   >>> model = AutoModelForCausalLM.from_pretrained(
   ...     "{OUTPUT_DIR / 'trained_model'}",
   ...     device_map="auto",
   ...     torch_dtype=torch.float16
   ... )
   
   # Анализ файла
   >>> result = predict_movie_ratings_improved("путь/к/сценарию.txt", model, tokenizer)
   >>> print(f"Рейтинг: {{result['predicted_rating']:.1f}}")
   >>> print(f"Возрастной рейтинг: {{result['predicted_age_rating']}}")

ПАРАМЕТРЫ МОДЕЛИ:
{json.dumps(config, indent=2, ensure_ascii=False)}

Дата создания: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

with open(OUTPUT_DIR / "README.md", "w", encoding=ENCODING) as f:
    f.write(usage_example)

print(f"\n📖 Инструкция сохранена в: {OUTPUT_DIR / 'README.md'}")

# Запускаем интерактивный режим
print("\n" + "="*50)
print("🚀 ЗАПУСК ИНТЕРАКТИВНОГО РЕЖИМА...")
print("="*50)

interactive_testing_improved()

CUDA: True
NVIDIA GeForce RTX 5070 Ti
Конфигурация загружена!
ROOT: C:\Users\Дмитрий\Downloads\Ratings
Загружено 51 записей из scripts_ratingss.csv

Первые 3 записи:
                    filename                  title  year  kp_rating  \
0  13_причин_почему_Кино.txt  13_причин_почему_Кино  2017        7.3   
1    Игра_Престолов_Кино.txt    Игра_Престолов_Кино  2011        9.0   
2     8_миллиметров_Кино.txt     8_миллиметров_Кино  1999        7.1   

   imdb_rating notes age_rating_imdb age_rating_kp           english_title  
0          7.4    ok           TV-MA           16+    TH1RTEEN R3ASONS WHY  
1          9.2    ok           TV-MA           18+  A Song of Ice and Fire  
2          6.6    ok               R           18+            8 Millimeter  
Обрезан длинный текст: 8_миллиметров_Кино.txt (100000 символов)
Обрезан длинный текст: Джокер_Кино.txt (100000 символов)
Обрезан длинный текст: Довод_Кино.txt (100000 символов)
Обрезан длинный текст: Железный_человек_Кино.txt (100000 сим

c:\Users\Дмитрий\Downloads\Ratings\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Подготовка датасетов...


Map: 100%|██████████| 8/8 [00:00<00:00, 262.87 examples/s]


Размеры датасетов:
Обучающий: 43 примеров
Валидационный: 8 примеров

НАСТРОЙКА МОДЕЛИ
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820

НАЧАЛО ОБУЧЕНИЯ


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.



✅ Модель сохранена в: C:\Users\Дмитрий\Downloads\Ratings\Out\trained_model

ТЕСТИРОВАНИЕ НА НОВЫХ ФАЙЛАХ (ТОЛЬКО NOINFO)

📂 Тестирование файлов из: C:\Users\Дмитрий\Downloads\Ratings\datasets\noinfo
Найдено файлов: 3

┌─ Файл 1/3: Анатидаефобия_Кино.txt
├─ Предсказанный рейтинг: ❌ Не определен
├─ Возрастной рейтинг: ❌ Не определен
└─ Ответ модели: ...

┌─ Файл 2/3: Прокрастинация_Кино.txt
├─ Предсказанный рейтинг: ❌ Не определен
├─ Возрастной рейтинг: ❌ Не определен
└─ Ответ модели: ...

┌─ Файл 3/3: Ширванская_сказка_Кино.txt
├─ Предсказанный рейтинг: ❌ Не определен
├─ Возрастной рейтинг: ❌ Не определен
└─ Ответ модели: ...

АНАЛИЗ РЕЗУЛЬТАТОВ ТЕСТИРОВАНИЯ

📊 СТАТИСТИКА:
Всего протестировано файлов: 3
Успешно определено рейтингов: 0/3 (0.0%)
Успешно определено возрастных рейтингов: 0/3 (0.0%)

💾 Результаты сохранены в: C:\Users\Дмитрий\Downloads\Ratings\Out\test_results.csv

🏆 ТОП-3 результата:

1. Анатидаефобия_Кино.txt
   Рейтинг: Нет
   Возрастной: Нет

2. Прокрастинация_Кино.txt


NameError: name 'json' is not defined